# ES 335 ML Project: 03 Price Variation Audit (Training Weeks Only)

Using **training weeks only**, this notebook computes price stability metrics per product:
1. Active sales weeks (`Weeks_With_Sales`)
2. Distinct weekly price levels (`Distinct_Weekly_Prices`)
3. Average weeks per price level (`Weeks_Per_Price`)
4. Price stability ratio: `Min_Price / Max_Price` (`Min_Max_Price_Ratio`)

It then applies customizable stability thresholds to filter candidate products for the what-if revenue tool.

In [1]:
import os
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.cleaning import load_raw_data
from src.pricing import get_training_split, compute_training_price_metrics, filter_products_by_thresholds

df_clean = load_raw_data(cache_path='../data/processed/data_clean.parquet')
# Partition strictly into training split (excluding holdout weeks)
train_df, test_df, split_date = get_training_split(df_clean, test_weeks_count=10)
print(f'Training records: {len(train_df):,} | Cutoff split date: {split_date}')

[data_loader] Loading cached cleaned data from '../data/processed/data_clean.parquet'...
[data_loader] Loaded 522,537 cleaned records from cache.
Training records: 359,150 | Cutoff split date: 2011-10-03 00:00:00


## 1. Compute Price Stability Metrics on Training Split

In [2]:
price_stats = compute_training_price_metrics(train_df)
print(f'Computed price metrics across {len(price_stats):,} unique products in training weeks.')
price_stats.head(15)

Computed price metrics across 3,739 unique products in training weeks.


,StockCode,Description,Total_Units_Train,Weeks_With_Sales,Distinct_Weekly_Prices,Weeks_Per_Price,Min_Price,Max_Price,Min_Max_Price_Ratio
0,23166,MEDIUM CERAMIC TOP STORAGE JAR,76917,22,18,1.22,1.04,2.06,0.505
1,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,41037,42,14,3.00,0.26,0.41,0.634
2,85099B,JUMBO BAG RED RETROSPOT,35952,43,33,1.30,1.94,3.14,0.618
3,22197,SMALL POPCORN HOLDER,32365,43,27,1.59,0.88,1.35,0.652
4,21212,PACK OF 72 RETROSPOT CAKE CASES,31252,43,25,1.72,0.54,0.90,0.600
5,85123A,WHITE HANGING HEART T-LIGHT HOLDER,30254,43,28,1.54,2.87,3.39,0.847
6,84879,ASSORTED COLOUR BIRD ORNAMENT,27085,43,18,2.39,1.64,1.93,0.850
7,15036,ASSORTED COLOURS SILK FAN,21898,43,33,1.30,0.82,1.50,0.547
8,21977,PACK OF 60 PINK PAISLEY CAKE CASES,21747,43,26,1.65,0.53,1.18,0.449
9,22616,PACK OF 12 LONDON TISSUES,20964,43,24,1.79,0.28,0.63,0.444


## 2. Filter Candidate Products by Adjustable Stability Thresholds

In [3]:
passed_products = filter_products_by_thresholds(
    price_stats,
    min_weeks_with_sales=20,
    min_total_units=500,
    min_price_ratio=0.50,
    min_weeks_per_price=1.0
)
print(f'Products meeting all stability criteria: {len(passed_products):,}')
passed_products.head(20)

Products meeting all stability criteria: 515


,StockCode,Description,Total_Units_Train,Weeks_With_Sales,Distinct_Weekly_Prices,Weeks_Per_Price,Min_Price,Max_Price,Min_Max_Price_Ratio
0,23166,MEDIUM CERAMIC TOP STORAGE JAR,76917,22,18,1.22,1.04,2.06,0.505
1,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,41037,42,14,3.00,0.26,0.41,0.634
2,85099B,JUMBO BAG RED RETROSPOT,35952,43,33,1.30,1.94,3.14,0.618
3,22197,SMALL POPCORN HOLDER,32365,43,27,1.59,0.88,1.35,0.652
4,21212,PACK OF 72 RETROSPOT CAKE CASES,31252,43,25,1.72,0.54,0.90,0.600
5,85123A,WHITE HANGING HEART T-LIGHT HOLDER,30254,43,28,1.54,2.87,3.39,0.847
6,84879,ASSORTED COLOUR BIRD ORNAMENT,27085,43,18,2.39,1.64,1.93,0.850
7,15036,ASSORTED COLOURS SILK FAN,21898,43,33,1.30,0.82,1.50,0.547
8,22492,MINI PAINT SET VINTAGE,19938,43,17,2.53,0.62,1.14,0.544
9,22178,VICTORIAN GLASS HANGING T-LIGHT,18159,43,35,1.23,1.21,2.06,0.587


### Key Takeaways from Price Variation Check
- A total of **515 products** pass the stability check criteria.
- When matched against our top 100 modeled volume panel, **62 products** passed the check.
- These products provide the empirical basis for generating candidate prices within $[P_5, P_{95}]$ in the What-If Revenue Tool.